# Preprocessing Decisions — When and Why Each Step Matters

*Notebook #2 in the hands-on MNE series. Assumes the material of notebook #1: loading `Raw` objects, computing PSDs, filtering, epoching, and computing ERPs.*

Notebook #1 presented the core EEG processing steps as a linear sequence: load → filter → epoch → average. That sequence is a useful scaffold, but it conceals a layer of decisions that determine whether the final result is trustworthy or misleading.

In practice, every step involves a choice: *Which* filter? *What* epoch window? *How strict* an artifact threshold? These are not arbitrary — each choice has consequences, and the correct choice depends on what you plan to do with the data downstream.

This notebook makes those decisions visible. Using the MNE sample dataset you already know, we explore four classes of preprocessing decisions through realistic scenarios. In each case, we compare the *right* choice to plausible alternatives and observe what goes wrong.

> **Pedagogical note.** As in the later notebooks, each scenario begins with a pause asking you to predict the outcome before seeing the code. These pauses are essential. The point is not to memorise parameter values but to build the intuition for *why* one choice is better than another in a given context.

## Table of contents

1. **Data loading** — The unfiltered MNE sample recording, inspected before any processing.
2. **Scenario A: The quality controller** — *"Is this recording usable?"* → Systematic quality assessment before processing.
3. **Scenario B: The filter dilemma** — *"How should I filter?"* → Different goals demand different filters; the wrong filter destroys the signal of interest.
4. **Scenario C: The epoching strategist** — *"How should I segment and clean my data?"* → Epoch window, baseline, and rejection threshold as interacting tradeoffs.
5. **Scenario D: The representation navigator** — *"Should I look at time or frequency?"* → The same data, two domains, different information.
6. **Synthesis** — A decision checklist for preprocessing.
7. **Practice scenarios** — Unsolved problems for self-assessment.

## Position in the textbook

- **Rao Ch. 3 — Recording Electrical Activity.** The physical process that produces the raw signal and its contaminants.
- **Rao Ch. 4 — Signal Processing.** Filtering, spectral analysis, and artifact handling as prerequisites to any analysis.
- **Rao Ch. 9.2 — Preprocessing.** The practical steps required before feature extraction in a BCI pipeline.

## 1. Data loading

We load the **unfiltered** version of the MNE sample dataset. This is important: starting from unfiltered data lets us observe problems (line noise, drift) that a pre-filtered file would hide. In practice, you will almost always receive unfiltered data from the recording system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import mne
from mne.datasets import sample

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

print("MNE:", mne.__version__)

In [ ]:
data_path = sample.data_path()
raw_fname = data_path / "MEG" / "sample" / "sample_audvis_raw.fif"

raw = mne.io.read_raw_fif(raw_fname, preload=True)

# Extract events before any channel manipulation.
events = mne.find_events(raw, stim_channel="STI 014")

# Keep EEG and EOG channels.
raw.pick(["eeg", "eog"])

print(f"Channels:      {len(raw.ch_names)} ({raw.info['nchan']} total)")
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Duration:      {raw.times[-1]:.1f} s")
print(f"Events found:  {len(events)}")

At this point, no processing has been applied. The data contains everything: brain signals, line noise, drift, blink artifacts, muscle artifacts. The question is: **what do you do first, and why?**

---

---

## 2. Scenario A — The quality controller

### The situation

You have just received a recording from the lab. Before investing time in analysis, you need to answer a basic question:

> *"Is this recording usable? Are there obvious problems that need to be addressed — or is the data so corrupted that it should be discarded?"*

This is the first thing any competent analyst does. Skipping this step and jumping straight to filtering or epoching is one of the most common mistakes in EEG practice — it means you may spend hours analysing data that was never usable in the first place.

### ❓ Pause — your prediction

What would you check? Name at least three concrete things you would inspect in a raw recording before deciding it is usable. Think about what you learned in notebook #1 about artifacts and noise sources.

---

### A quality assessment protocol

A systematic quality check involves at least four inspections:

1. **Power spectrum** — Does the PSD look physiologically plausible? Is there a visible alpha peak (~10 Hz)? Is there line noise (50 or 60 Hz depending on the country)?
2. **Channel-level inspection** — Are any channels flat, excessively noisy, or drifting? These are "bad channels" that must be interpolated or excluded.
3. **Time-domain survey** — Are there gross artifacts (subject movement, electrode pops, amplifier saturation) that occupy large portions of the recording?
4. **EOG channel** — Does the EOG show regular blinks? This is actually *good* — it means blinks are captured and can later be removed with ICA.

### Check 1 — Power spectrum

In [ ]:
# Compute and plot the power spectral density (PSD) up to 100 Hz.
# We look at EEG channels only.
spectrum = raw.compute_psd(picks="eeg", fmax=100)
fig = spectrum.plot(average=True)
plt.title("Power spectrum — unfiltered recording (EEG average)")
plt.show()

### Interpretation — what to look for in the PSD

A healthy EEG PSD in an awake adult should show:

- A **1/f trend**: power decreases with frequency, roughly following a power law. If the spectrum is flat or has an unusual shape, something is wrong with the recording.
- An **alpha peak** near 8–12 Hz, visible as a "bump" above the 1/f slope. This is the most reliable marker that you are looking at real brain data.
- **Line noise**: a sharp spike at the mains frequency (60 Hz in the US, 50 Hz in Europe). The MNE sample data was recorded in the US, so expect 60 Hz. This is not a problem — it is removed by notch filtering — but its *absence* in US data would suggest the recording has already been processed.

Red flags:
- A completely flat spectrum (amplifier malfunction).
- Extremely high broadband power above 30 Hz (sustained muscle artifact).
- No alpha peak (possible in some individuals, but worth noting).

### Check 2 — Per-channel power spectrum

The average PSD can hide a single bad channel. Inspecting individual channels reveals outliers.

In [ ]:
# Per-channel PSD — look for channels that deviate strongly from the group.
fig = spectrum.plot(average=False, amplitude=False)
plt.title("Per-channel power spectra")
plt.show()

In [ ]:
# Quantitative bad-channel detection: compute the total power for each channel
# and flag those that deviate more than 3 standard deviations from the median.
psd_data = spectrum.get_data()  # shape: (n_channels, n_freqs)
total_power = psd_data.sum(axis=1)

median_power = np.median(total_power)
mad = np.median(np.abs(total_power - median_power))  # median absolute deviation
threshold = median_power + 3 * 1.4826 * mad  # 1.4826 converts MAD to σ for Gaussian

eeg_names = spectrum.ch_names
suspect = [eeg_names[i] for i in range(len(eeg_names)) if total_power[i] > threshold]

if suspect:
    print(f"Potentially bad channels (power > 3 MAD-σ from median): {suspect}")
else:
    print("No channels flagged as outliers — all within expected range.")

### Check 3 — Peak-to-peak amplitude over time

A useful summary of overall signal quality is the peak-to-peak amplitude computed in short sliding windows. Periods with very high amplitude suggest gross artifacts (movement, electrode pop); periods with very low amplitude suggest electrode disconnection.

In [ ]:
# Compute peak-to-peak amplitude in 1-second windows across all EEG channels.
eeg_data = raw.copy().pick("eeg").get_data()  # (n_channels, n_samples)
sfreq = raw.info["sfreq"]
win_samples = int(sfreq)  # 1-second window

n_windows = eeg_data.shape[1] // win_samples
ptp_per_window = np.zeros(n_windows)

for i in range(n_windows):
    segment = eeg_data[:, i * win_samples : (i + 1) * win_samples]
    ptp_per_window[i] = np.median(np.ptp(segment, axis=1))  # median across channels

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.arange(n_windows), ptp_per_window * 1e6, color="steelblue", linewidth=0.8)
ax.axhline(200, color="red", linestyle="--", alpha=0.5, label="200 µV (typical concern threshold)")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Median peak-to-peak (µV)")
ax.set_title("Signal amplitude over time — screening for gross artifacts")
ax.legend()
plt.tight_layout()
plt.show()

bad_windows = np.sum(ptp_per_window * 1e6 > 200)
print(f"Windows exceeding 200 µV: {bad_windows} / {n_windows} ({100*bad_windows/n_windows:.1f}%)")

### Quality verdict

Based on the three checks above, you should now be able to answer:

- **Is the recording physiologically plausible?** (1/f slope, alpha peak: yes/no)
- **Are there bad channels that need attention?** (list them)
- **What fraction of the recording is contaminated by gross artifacts?** (percentage of high-amplitude windows)

In a real workflow, this assessment takes 2–3 minutes and can save hours of wasted analysis. A recording where >30% of windows are grossly contaminated, or where multiple channels are bad, may not be worth processing.

---

❓ **Exercise.** The alpha peak in the average PSD is generated primarily by posterior (occipital) channels. Use `spectrum.plot_topomap()` with `bands={"Alpha": (8, 12)}` to verify this. If the alpha peak were frontal rather than posterior, what might that indicate about the recording?

---

## 3. Scenario B — The filter dilemma

### The situation

Quality assessment is complete; the recording is usable. You are now ready to filter — but which filter? In notebook #1, the filter was presented as a fixed recipe: band-pass 1–40 Hz. In practice, the choice is not fixed. Consider three researchers, each planning a different analysis:

- **Researcher 1** wants to study **slow cortical potentials** (SCPs) — brain responses that unfold over 1–2 seconds, at frequencies below 1 Hz.
- **Researcher 2** wants to study the **auditory N100** — a fast ERP component peaking near 100 ms.
- **Researcher 3** wants to run **ICA** to remove eye artifacts, then study ERPs.

> *"Should all three use the same filter? If not, why not?"*

### ❓ Pause — your prediction

1. Researcher 1 studies signals below 1 Hz. What happens to their signal of interest if they high-pass filter at 1 Hz?
2. Researcher 3 wants to run ICA. MNE's ICA documentation recommends a high-pass filter of at least 1 Hz for stable decomposition. Is this compatible with Researcher 1's needs? How would you resolve the conflict?
3. All three want to study ERPs below 40 Hz. Is the low-pass cutoff (40 Hz) uncontroversial, or does it also depend on the question?

---

### Demonstration — the same data, three different filters

We apply three filter settings to the same recording and compare the consequences.

In [ ]:
# Three filter configurations.
raw_hp01 = raw.copy().filter(l_freq=0.1, h_freq=40.0)   # Preserves slow components
raw_hp1  = raw.copy().filter(l_freq=1.0, h_freq=40.0)    # Standard for ICA / fast ERPs
raw_hp2  = raw.copy().filter(l_freq=2.0, h_freq=40.0)    # Aggressive — may distort ERPs

print("Filtering complete.")

In [ ]:
# Compare the three filter settings on a single channel time segment.
ch_name = "EEG 050"
ch_idx_raw = raw.ch_names.index(ch_name)
start, stop = int(20 * sfreq), int(30 * sfreq)  # 10-second window
t = np.arange(stop - start) / sfreq + 20

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)

for ax, r, label in zip(axes, [raw, raw_hp01, raw_hp1, raw_hp2],
                         ["Unfiltered", "HP 0.1 Hz + LP 40 Hz",
                          "HP 1.0 Hz + LP 40 Hz", "HP 2.0 Hz + LP 40 Hz"]):
    data = r.get_data(picks=[ch_name])[0, start:stop]
    ax.plot(t, data * 1e6, linewidth=0.6, color="steelblue")
    ax.set_ylabel("µV")
    ax.set_title(label, fontsize=10)
    ax.set_ylim(-60, 60)

axes[-1].set_xlabel("Time (s)")
plt.suptitle(f"Effect of high-pass filter cutoff on {ch_name}", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### The critical test — how do these filters affect the ERP?

The time-domain traces above show differences in slow drift, but the real question is: **do these filters change the ERP itself?** If the high-pass cutoff distorts the evoked waveform, the clinician from notebook #3 would draw incorrect conclusions.

In [ ]:
# Epoch and average under each filter setting.
event_id = {"auditory/left": 1, "auditory/right": 2}

evokeds = {}
for label, r in [("HP 0.1 Hz", raw_hp01), ("HP 1.0 Hz", raw_hp1), ("HP 2.0 Hz", raw_hp2)]:
    r_eeg = r.copy().pick("eeg")
    r_eeg.set_eeg_reference("average", projection=True)
    r_eeg.apply_proj()
    ep = mne.Epochs(r_eeg, events, event_id, tmin=-0.2, tmax=0.5,
                    baseline=(None, 0), preload=True, reject=dict(eeg=100e-6))
    evokeds[label] = ep.average()

# Compare the ERPs at a frontocentral channel.
ch = "EEG 050"
fig, ax = plt.subplots(figsize=(10, 4))
for label, ev in evokeds.items():
    idx = ev.ch_names.index(ch)
    ax.plot(ev.times * 1000, ev.data[idx] * 1e6, label=label, linewidth=1.5)

ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title(f"Auditory ERP at {ch} — effect of high-pass cutoff")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation — the filter tradeoff

You should observe that the three ERPs are **not identical**. The aggressive high-pass (2 Hz) introduces subtle distortions: the waveform may show an artificial undershoot before stimulus onset or a shift in apparent peak latency. This is a well-documented phenomenon — high-pass filtering can create **filter ringing** that mimics or distorts ERP components.

The tradeoff:

| High-pass cutoff | Advantage | Risk |
|---|---|---|
| **0.1 Hz** | Preserves all slow components, including SCPs. Minimal ERP distortion. | Retains slow drifts that complicate baseline correction and ICA. |
| **1.0 Hz** | Good compromise. Removes most drift. Recommended for ICA fitting. | Attenuates signals below 1 Hz (SCPs, Contingent Negative Variation). |
| **2.0 Hz** | Removes virtually all drift. Very clean baseline. | Can distort ERP morphology — may shift apparent latencies and introduce artifactual peaks. |

The correct choice depends on the downstream analysis. There is no universally "right" filter.

📖 **Practical recommendation.** A common workflow for ERP studies that require ICA: (1) filter at 1 Hz, (2) fit ICA, (3) apply the ICA solution to the *0.1 Hz-filtered* data. This gives ICA a clean signal for decomposition while preserving slow components in the final data.

---

❓ **Exercise.** The MNE sample data was recorded in the US, where mains frequency is 60 Hz. If the recording were from Europe (50 Hz), what would you change in the notch filter? What if you were unsure about the recording location — how would the PSD tell you which line noise frequency to notch?

---

## 4. Scenario C — The epoching strategist

### The situation

You have decided on a filter (1 Hz high-pass for this example). Now you must epoch the data. In notebook #1, epoching was presented with fixed parameters: `tmin=-0.2`, `tmax=0.5`, `reject=150e-6`. But each of these values is a decision with consequences.

> *"What epoch window should I use? How long should the baseline be? How strict should artifact rejection be?"*

### ❓ Pause — your prediction

1. If you set the epoch window too short (e.g., 0 to 200 ms), what information might you lose?
2. If you set the rejection threshold too strict (e.g., 50 µV), you will reject many epochs. If too lenient (e.g., 500 µV), you keep noisy epochs. What is the cost of each error?
3. The baseline is the pre-stimulus interval whose mean is subtracted from each epoch. What happens if the baseline is too short (e.g., 10 ms) or if it contains an artifact?

---

### Demonstration — the rejection threshold tradeoff

We epoch the same data with three different rejection thresholds and observe how each affects the number of retained epochs and the quality of the resulting ERP.

In [ ]:
# Use the 1 Hz-filtered data with average reference.
raw_proc = raw_hp1.copy().pick("eeg")
raw_proc.set_eeg_reference("average", projection=True)
raw_proc.apply_proj()

event_id_all = {
    "auditory/left": 1, "auditory/right": 2,
    "visual/left": 3,  "visual/right": 4,
}

thresholds = [50e-6, 100e-6, 300e-6]  # µV, converted to V
labels_thr = ["50 µV (strict)", "100 µV (moderate)", "300 µV (lenient)"]

results = []
for thr, lab in zip(thresholds, labels_thr):
    ep = mne.Epochs(raw_proc, events, event_id_all,
                    tmin=-0.2, tmax=0.5, baseline=(None, 0),
                    preload=True, reject=dict(eeg=thr))
    n_total = len(events[np.isin(events[:, -1], list(event_id_all.values()))])
    n_kept = len(ep)
    results.append((lab, thr, n_kept, n_total))
    print(f"{lab:22s}: {n_kept:3d} / {n_total} epochs retained ({100*n_kept/n_total:.0f}%)")

In [ ]:
# Compare the auditory ERPs produced by each threshold.
ch = "EEG 050"
fig, ax = plt.subplots(figsize=(10, 4))

colors = ["#e74c3c", "#2980b9", "#27ae60"]
for (lab, thr, n_kept, _), col in zip(results, colors):
    ep = mne.Epochs(raw_proc, events, {"auditory/left": 1, "auditory/right": 2},
                    tmin=-0.2, tmax=0.5, baseline=(None, 0),
                    preload=True, reject=dict(eeg=thr))
    ev = ep.average()
    idx = ev.ch_names.index(ch)
    ax.plot(ev.times * 1000, ev.data[idx] * 1e6,
            label=f"{lab} (n={len(ep)})", linewidth=1.5, color=col)

ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title(f"Auditory ERP at {ch} — effect of rejection threshold")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation — the rejection tradeoff

You should observe a tradeoff:

- **Strict threshold (50 µV):** Very few epochs survive. The ERP is clean but may be unreliable due to the small number of trials — and the surviving trials may represent a biased subset (e.g., only trials where the subject was very still).
- **Lenient threshold (300 µV):** Most epochs survive, but the ERP is noisier because artifact-contaminated trials are included in the average.
- **Moderate threshold (100 µV):** A compromise — enough trials for a reliable average, without gross artifact contamination.

The "right" threshold depends on:
- **How many trials you have.** If the experiment provides 500 trials, you can afford strict rejection. If you have only 30 (e.g., infant or clinical data), you cannot.
- **How much noise you can tolerate.** BCI classification may be robust to some noise; a clinical N100 measurement may not.
- **Whether you plan to use ICA.** If ICA will remove ocular artifacts, you can set a higher threshold (or skip amplitude-based rejection entirely) because blinks are handled separately.

### Demonstration — the epoch window choice

The epoch window determines how much temporal context is available. Too short and you miss late components; too long and you waste data or include activity from adjacent trials.

In [ ]:
# Compare three epoch windows.
windows = [(-0.05, 0.2), (-0.2, 0.5), (-0.5, 1.0)]
labels_win = ["−50 to 200 ms (narrow)", "−200 to 500 ms (standard)",
              "−500 to 1000 ms (wide)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (tmin, tmax), lab in zip(axes, windows, labels_win):
    ep = mne.Epochs(raw_proc, events, {"auditory/left": 1, "auditory/right": 2},
                    tmin=tmin, tmax=tmax, baseline=(tmin, 0),
                    preload=True, reject=dict(eeg=100e-6))
    ev = ep.average()
    idx = ev.ch_names.index(ch)
    ax.plot(ev.times * 1000, ev.data[idx] * 1e6, linewidth=1.5, color="steelblue")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Time (ms)")
    ax.set_title(lab, fontsize=10)
    ax.set_xlim(tmin * 1000, tmax * 1000)

axes[0].set_ylabel("Amplitude (µV)")
plt.suptitle(f"Auditory ERP at {ch} — effect of epoch window", fontsize=12)
plt.tight_layout()
plt.show()

### Interpretation — epoch window

- **Narrow window (−50 to 200 ms):** The N100 is visible, but later components (P200, N200) are clipped. The very short baseline (50 ms) provides an unreliable estimate of the pre-stimulus mean, making baseline correction noisy.
- **Standard window (−200 to 500 ms):** Captures the full ERP complex (N100, P200, and any later components up to 500 ms). The 200 ms baseline is sufficient for stable correction. This is the most common choice for fast ERP studies.
- **Wide window (−500 to 1000 ms):** Captures very late components and allows time-frequency analysis at low frequencies (wavelets need temporal margin). However, adjacent trials may overlap if the inter-stimulus interval is short, and more epochs may be rejected because the longer window has a higher probability of containing an artifact.

📖 **Rule of thumb.** The epoch window should be as long as your question requires, and no longer. If you study the N100, you do not need 1000 ms of post-stimulus data. If you study time-frequency decomposition of alpha blocking, you do.

---

❓ **Exercise.** In the narrow window, the baseline is only 50 ms (a few dozen samples at 600 Hz). Compute the standard error of the mean for a 50 ms baseline versus a 200 ms baseline. How much noisier is the baseline correction with the short window? (Hint: standard error scales as 1/√n.)

---

## 5. Scenario D — The representation navigator

### The situation

Your preprocessing is complete: the data is filtered, epoched, and cleaned. You are about to begin analysis — but before you do, you face a fundamental question:

> *"Should I examine this data in the time domain or the frequency domain? What does each representation reveal, and what does each hide?"*

This is not a preprocessing question — it is a *representation* question. The same epochs can be viewed as waveforms (voltage vs time) or as spectra (power vs frequency), and each view tells a different story.

### ❓ Pause — your prediction

1. The auditory N100 is a brief, sharp deflection. Would it be visible in the power spectrum of an epoch?
2. The alpha rhythm is an ongoing 10 Hz oscillation. Would it be visible in the trial-averaged ERP?
3. Can you think of a signal that is visible in *both* domains?

---

In [ ]:
# Use the standard-window epochs with moderate rejection.
epochs_demo = mne.Epochs(raw_proc, events, {"auditory/left": 1, "auditory/right": 2},
                         tmin=-0.2, tmax=0.5, baseline=(None, 0),
                         preload=True, reject=dict(eeg=100e-6))

evoked_demo = epochs_demo.average()
print(f"Epochs: {len(epochs_demo)}")

In [ ]:
# Side-by-side: time domain vs frequency domain for the same auditory epochs.
ch = "EEG 050"
ch_idx = evoked_demo.ch_names.index(ch)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Left: time domain (ERP)
ax1.plot(evoked_demo.times * 1000, evoked_demo.data[ch_idx] * 1e6,
         color="steelblue", linewidth=1.5)
ax1.axhline(0, color="grey", linewidth=0.5)
ax1.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax1.set_xlabel("Time (ms)")
ax1.set_ylabel("Amplitude (µV)")
ax1.set_title(f"Time domain — Auditory ERP at {ch}")

# Right: frequency domain (PSD of the epochs)
spectrum_epochs = epochs_demo.compute_psd(fmin=1, fmax=40, picks=[ch])
psds = spectrum_epochs.get_data().mean(axis=0)  # average across epochs
freqs_psd = spectrum_epochs.freqs
ax2.semilogy(freqs_psd, psds[0], color="steelblue", linewidth=1.5)
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("Power (V²/Hz)")
ax2.set_title(f"Frequency domain — PSD at {ch}")

plt.suptitle("Same data, two representations", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Interpretation — what each domain reveals

| Feature | Time domain (ERP) | Frequency domain (PSD) |
|---|---|---|
| **N100** | Clearly visible as a negative peak at ~100 ms | Invisible — a brief transient does not produce a spectral peak |
| **Alpha rhythm** | Invisible in the average (cancelled by phase variability) | Visible as a peak near 10 Hz |
| **Noise level** | Visible as waveform roughness | Visible as the overall power level |
| **Drift** | Visible as slow baseline wandering | Visible as elevated low-frequency power |

The two domains are mathematically equivalent (the Fourier transform is invertible), but they make different features perceptually salient. Choosing the wrong domain does not lose information — it *hides* it.

📖 **When to use each domain:**
- **Time domain** when your question is about the *timing* and *morphology* of a response (ERP latency, waveform shape, temporal order of components).
- **Frequency domain** when your question is about the *rhythmic content* of the signal (which oscillatory frequencies are present, how strong they are, whether they change between conditions).
- **Time-frequency domain** (as in notebook #4, Scenario D) when you need *both* — how oscillatory power changes over time relative to an event.

### Topographic contrast — where to look in each domain

The spatial distribution of the signal differs depending on which domain you examine.

In [ ]:
# Topography in the time domain: the N100 peak.
fig_time = evoked_demo.plot_topomap(
    times=[0.10], ch_type="eeg", time_unit="s", colorbar=True,
)
plt.suptitle("Time domain — topography at 100 ms (N100)", y=1.05)
plt.show()

In [ ]:
# Topography in the frequency domain: alpha-band (8–12 Hz) power.
spectrum_all = epochs_demo.compute_psd(fmin=1, fmax=40)
fig_freq = spectrum_all.plot_topomap(bands={"Alpha (8-12 Hz)": (8, 12)},
                                      ch_type="eeg")
plt.suptitle("Frequency domain — alpha-band power topography", y=1.05)
plt.show()

### Interpretation — spatial distribution

The N100 (time domain) is maximal over **fronto-central** scalp — consistent with generators in the auditory cortex (superior temporal gyrus). Alpha power (frequency domain) is maximal over **posterior** scalp — consistent with generators in the occipital cortex.

A researcher who only examines the frequency domain would never see the N100. A researcher who only examines the time domain (via averaging) would never see the alpha rhythm. **Choosing the representation is choosing what you can see.**

---

❓ **Exercise.** There is a third representation you have not used yet: the **time-frequency** representation, which shows power as a function of both time and frequency. In what scenario would you need this representation rather than either of the two shown above? (Hint: think about event-related changes in oscillatory power, as discussed in notebook #1's section on brain rhythms.)

---

## 6. Synthesis — a preprocessing decision checklist

Before running any analysis on a new dataset, work through the following checklist. Each item is a decision, not a recipe.

### Step 1 — Quality assessment (Scenario A)

- Inspect the PSD. Is the 1/f slope present? Is there an alpha peak? Is there line noise?
- Check for bad channels (flat, excessively noisy, or drifting).
- Survey the time domain for gross artifacts.
- **Decision:** Proceed, interpolate bad channels, or discard the recording.

### Step 2 — Filtering (Scenario B)

- What is the lowest frequency your analysis requires? Set the high-pass accordingly.
- Will you run ICA? If so, use ≥1 Hz for the ICA fit, then apply the solution to more gently filtered data.
- Apply a notch filter for line noise (50 or 60 Hz, confirmed by PSD inspection).
- **Decision:** Choose the high-pass and low-pass cutoffs based on your downstream question.

### Step 3 — Epoching (Scenario C)

- What is the time window your analysis requires? Include only as much as needed.
- Provide a baseline of at least 100–200 ms for stable baseline correction.
- Set the rejection threshold based on how many trials you can afford to lose.
- **Decision:** Choose the epoch window, baseline, and rejection threshold as an interacting set of tradeoffs.

### Step 4 — Representation (Scenario D)

- Is your question about timing and waveform shape? → Time domain.
- Is your question about rhythmic content? → Frequency domain.
- Is your question about how oscillatory power changes over time? → Time-frequency domain.
- **Decision:** Choose the representation before the analysis, not after.

---

## 7. Practice scenarios

For each scenario, identify the correct preprocessing decisions. Try to answer before checking the solution.

### Scenario P1

> You are recording EEG in a classroom to study sustained attention. The recording will last 45 minutes. Electrode gel tends to dry during long recordings, leading to increasing impedance and slow drift. What filter settings and quality-monitoring strategy would you choose?

<details>
<summary>Click to reveal the analysis</summary>

- **Quality:** Monitor impedance at regular intervals. Check the PSD at the beginning, middle, and end of the recording for changes in noise level. Use the peak-to-peak amplitude plot to identify when signal quality degrades.
- **Filtering:** A 0.5–1.0 Hz high-pass is advisable because long recordings accumulate severe slow drift. If you study ERPs, this is acceptable. If you study slow cortical potentials, you face a conflict between drift removal and signal preservation.
- **Epoching:** With a 45-minute recording, you will have hundreds of events. You can afford strict rejection (e.g., 80–100 µV) because losing 20–30% of trials still leaves enough for reliable averaging.
</details>

### Scenario P2

> You receive a clinical EEG recording from a hospital. The file contains 19 channels (the standard 10-20 montage). One channel (T3) appears flat — no signal at all. What should you do, and what should you NOT do?

<details>
<summary>Click to reveal the analysis</summary>

- **Do:** Mark T3 as a bad channel (`raw.info['bads'] = ['T3']`). After preprocessing, interpolate it using the surrounding channels (`raw.interpolate_bads()`). This reconstructs a plausible signal at that location.
- **Do NOT:** Simply ignore it or include it in the average reference — a flat channel in the average reference will bias all other channels. Also do not delete it silently, as downstream analyses (e.g., source localisation) may expect the full 10-20 montage.
- **Report it.** Note in your analysis log that T3 was interpolated. This is essential for reproducibility.
</details>

### Scenario P3

> You are studying the Contingent Negative Variation (CNV) — a slow negative ERP that builds up over 1–2 seconds between a warning stimulus and a target. Your collaborator suggests high-pass filtering at 1 Hz. Is this a good idea?

<details>
<summary>Click to reveal the analysis</summary>

- **No.** The CNV is a slow cortical potential with most of its energy below 1 Hz. A 1 Hz high-pass would severely attenuate or eliminate the signal of interest. You need a high-pass of at most 0.01–0.05 Hz, or no high-pass at all (DC recording).
- **The tradeoff:** Without a high-pass, slow drifts will be severe, making baseline correction and ICA difficult. You may need to handle drift with detrending (polynomial or linear regression) rather than filtering, or accept that ICA will not work optimally and rely on manual artifact rejection.
- **This is Scenario B, Research 1's problem:** the downstream question determines the filter, not the other way around.
</details>

### Scenario P4

> You are analysing data from an epilepsy patient. The neurologist asks you to identify epileptiform discharges ("spikes") in the raw EEG. Should you epoch the data around stimulus events and average, as in an ERP study?

<details>
<summary>Click to reveal the analysis</summary>

- **No.** Epileptiform discharges are spontaneous events — they are not time-locked to any external stimulus. Epoching around stimulus markers is meaningless here.
- **Approach:** This requires continuous (non-epoched) analysis. You inspect the raw time series, possibly with automated spike-detection algorithms, looking for characteristic morphology (sharp waves, spike-and-wave complexes) at any time during the recording.
- **Domain:** Primarily the time domain — the spike morphology is the diagnostic feature. Frequency-domain analysis may supplement (epileptiform activity can produce broadband power increases), but the clinical diagnosis is made from the waveform shape.
- **Lesson:** Not all EEG analysis involves epoching. The decision to epoch depends on whether the events of interest are time-locked to known markers.
</details>

---

## 8. Key takeaways

1. **Always assess quality first.** Two minutes of PSD inspection and amplitude screening can save hours of processing data that was never usable.

2. **There is no universally correct filter.** The high-pass cutoff determines which frequencies survive. A 1 Hz high-pass is appropriate for fast ERPs and ICA; it destroys slow cortical potentials. A 0.1 Hz high-pass preserves slow signals but retains drift. The downstream question determines the filter.

3. **Epoch parameters interact.** The epoch window, baseline duration, and rejection threshold are not independent choices. A short baseline means noisy correction. A strict threshold means fewer trials. A long window means more artifact exposure. These must be balanced as a system.

4. **Choosing a representation is choosing what you can see.** The time domain reveals waveform morphology and timing. The frequency domain reveals rhythmic content. Neither is "better" — they answer different questions. Choosing the wrong representation does not produce a wrong answer; it produces *no* answer.

---

## 9. What comes next

- **Notebook #3 — Motor Imagery BCIs.** The first complete BCI pipeline, applying the preprocessing decisions from this notebook to a motor-imagery task.
- **Notebook #4 — From Research Question to Analysis Strategy.** A notebook that extends the "question first" approach from preprocessing to analysis: given the same preprocessed data, how does the research question determine whether you average, classify, decode over time, or decompose into time-frequency representations?